# 练习解答

把 Day 1 的网页摘要项目升级为使用通过 Ollama 在本地运行的开源模型，而不是 OpenAI

如果你不想使用付费 API，后续所有项目都可以使用这种技术。

**优点：**
1. 无 API 费用——开源
2. 数据不会离开你的电脑

**缺点：**
1. 能力明显弱于前沿模型（Frontier Model）

## Ollama 安装回顾

只需访问 [ollama.com](https://ollama.com) 并安装！

完成后，ollama 服务器应该已经在本地运行。  
如果你访问：  
[http://localhost:11434/](http://localhost:11434/)

你应该看到消息 `Ollama is running`。  

如果没有，打开新的 Terminal（Mac）或 Powershell（Windows）并输入 `ollama serve`  
然后再次尝试 [http://localhost:11434/](http://localhost:11434/)。

In [ ]:
# 导入

# 导入 requests：用 HTTP 发网络请求（比手写 urllib 更简单）
import requests
# 从 bs4 导入 BeautifulSoup：解析 HTML 网页，方便提取标题和正文
from bs4 import BeautifulSoup
# 从 IPython.display 导入展示工具：在 Jupyter 笔记本里漂亮地显示 Markdown/图片等
from IPython.display import Markdown, display
import ollama

In [ ]:
# 常量

# 选定本次实验使用的模型名称（model id）
MODEL = "llama3.2"

In [ ]:
# 表示网页的类

class Website:
    """
    A utility class to represent a Website that we have scraped
    """
    url: str
    title: str
    text: str

    # 定义函数：把一组步骤打包，方便重复调用
    def __init__(self, url):
        """
        Create this Website object from the given url using the BeautifulSoup library
        """
        self.url = url
        response = requests.get(url)
        soup = BeautifulSoup(response.content, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        self.text = soup.body.get_text(separator="\n", strip=True)

In [ ]:
# 让我们试一个

ed = Website("https://edwarddonner.com")
print(ed.title)
print(ed.text)

## 提示词的类型

你可能已经知道了——如果还不知道，很快就会非常熟悉！

像 GPT4o 这样的模型被训练成以特定方式接收指令。

它们期望收到：

**系统提示词（system prompt）**，告诉它们正在执行什么任务、应使用什么语气

**用户提示词（user prompt）**——它们应回复的对话开场白

In [ ]:
# 定义我们的系统提示词——你可以稍后实验，把最后一句改成 'Respond in markdown in Spanish."

system_prompt = "You are an assistant that analyzes the contents of a website \
and provides a short summary, ignoring text that might be navigation related. \
Respond in markdown."

In [ ]:
# 编写用户提示词、请求网站摘要的函数：

def user_prompt_for(website):
    # 用户提示词（user prompt）：本次要模型完成的具体任务与输入内容
    user_prompt = f"You are looking at a website titled {website.title}"
    user_prompt += "The contents of this website is as follows; \
please provide a short summary of this website in markdown. \
If it includes news or announcements, then summarize these too.\n\n"
    user_prompt += website.text
    return user_prompt

## 消息（Messages）

Ollama 的 API 期望与 OpenAI 相同的消息格式：

```
[
    {"role": "system", "content": "system message goes here"},
    {"role": "user", "content": "user message goes here"}
]

In [ ]:
# 看看这个函数如何精确创建上面的格式

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]

## 是时候把它们整合起来了——现在用 Ollama 代替 OpenAI

In [ ]:
# 现在：调用 Ollama 函数而不是 OpenAI

def summarize(url):
    # 抓取/构造网站内容，作为后续提示词的输入
    website = Website(url)
    # 组装 messages 列表：Chat Completions API 要求的对话格式（system / user / assistant）
    messages = messages_for(website)
    response = ollama.chat(model=MODEL, messages=messages)
    return response['message']['content']

In [ ]:
summarize("https://edwarddonner.com")

In [ ]:
# 用 markdown 在 Jupyter 输出中美观地显示结果的函数

def display_summary(url):
    # 拿到模型生成的摘要文本
    summary = summarize(url)
    # 用 Markdown 在笔记本中渲染格式化文本（display_id 方便后续原地刷新）
    display(Markdown(summary))

In [ ]:
display_summary("https://edwarddonner.com")

# 让我们试试更多网站

注意：这只适用于可以用这种简单方式抓取的网站。

用 Javascript 渲染的网站（如 React 应用）不会显示内容。请查看 community-contributions 文件夹中的 Selenium 实现来绕过此限制。你需要了解如何安装 Selenium（可以问 ChatGPT！）

另外，受 CloudFront（及类似服务）保护的网站可能会返回 403 错误——非常感谢 Andy J 指出这一点。

但许多网站都能正常工作！

In [ ]:
display_summary("https://cnn.com")

In [ ]:
display_summary("https://anthropic.com")

# 分享你的代码

我很希望你之后分享代码，这样我就能分享给其他人！你会注意到一些学员已经做了改动（包括 Selenium 实现），可以在 community-contributions 文件夹中找到。如果想把你的改动加入该文件夹，请提交包含新版本的 Pull Request，我会合并你的更改。

如果你不是 git 专家（我也不是！），GPT 给出了一些不错的关于如何提交 Pull Request 的说明。过程有点繁琐，但做过一次就很清楚了。小贴士：最好清空 Jupyter notebook 的输出（Edit >> Clean outputs of all cells，然后 Save），以保持 notebook 干净。

一位 AI 朋友提供的 PR 说明：https://chatgpt.com/share/670145d5-e8a8-8012-8f93-39ee4e248b4c